# Real-Trace Verdict for Admission Control\n\nThis notebook is a runnable demo of `eval.py`, which computes a pre-registered statistical verdict for **conformal admission control (ACI)** against four baseline admission policies, evaluated on **real Azure Functions trace data** (210,000 rows across 5 traffic regimes: `stationary`, `burst`, `drift`, `regime_switch`, `adversarial`).\n\nThe original experiment step (`gen_art_experiment_1`) was empty when this evaluation ran, so — as documented in the original script's docstring — the 5 admission policies (`conformal_aci`, `fixed_threshold`, `index_based`, `rl_frozen`, `oracle_hindsight`) are implemented directly in the evaluation code and run against the real trace data. For each `(policy, regime)` cell the script computes:\n\n- a rolling **admitted-request violation rate** (window of `WINDOW` admitted requests),\n- the post-burn-in **mean absolute deviation (MAD)** from the target `alpha`, with a pass/fail tolerance flag,\n- the **max transient spike**,\n- over-seed **bootstrap 95% CIs** and **Holm-Bonferroni-corrected** paired significance tests (conformal vs. each baseline).\n\nThis demo runs the *same code* on a small 100-row subset (20 rows per regime) with shrunk config values (window size, seed count, bootstrap draws) so it finishes in seconds instead of the full run's longer wall-clock time. The code itself — the policies, the rolling-stat math, the bootstrap/Holm logic — is unchanged from the original `eval.py`; only the input size and a few tunable constants are reduced.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru -- not pre-installed on Colab, always install
_pip('loguru==0.7.3')

# numpy, matplotlib -- pre-installed on Colab, install locally only (to match Colab's exact versions)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'matplotlib==3.10.0')

In [ ]:
# --- Imports (copied from the original eval.py's import block, plus matplotlib for viz) ---
from __future__ import annotations

import gc
import json
import time
from typing import Any

import numpy as np
from loguru import logger
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-papers/ai-invention-db8806-conformal-admission-control-distribution/main/round-2/evaluation-1/demo/mini_demo_data.json"
import os

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print("regimes present:", sorted({json.loads(e["input"])["regime_label"] for e in data["datasets"][0]["examples"]}))
print("total rows:", len(data["datasets"][0]["examples"]))

## Config\n\nAll tunable parameters from the original `eval.py`, collected in one place. `WINDOW`, `BURN_IN_MIN`, and `LOAD_WINDOW` are shrunk to the **minimum that still produces meaningful output** on this notebook's 100-row (20 per regime) demo subset — the original full-scale values (`WINDOW=500`, `BURN_IN_MIN=500`, `LOAD_WINDOW=50`) are commented alongside each line for reference and are what the real 210,000-row run used. `N_SEEDS`, `N_BOOTSTRAP`, and `ETA_GRID` are cheap regardless of data size, so they are kept at (or close to) their original values.

In [ ]:
ALPHA = 0.10
WINDOW = 5           # rolling window, in ADMITTED requests (original: 500)
BURN_IN_MIN = 2      # (original: 500)
BURN_IN_FRAC = 0.05
TOL_PP = 0.03
N_SEEDS = 5          # >=5 required for a genuine over-seed bootstrap (matches original spec)
N_BOOTSTRAP = 1000   # whole-seed resample draws (original: 10_000)
CALIB_SEED = 777
REGIMES = ["stationary", "burst", "drift", "regime_switch", "adversarial"]
BASELINES = ["fixed_threshold", "index_based", "rl_frozen"]
ALL_POLICIES = ["conformal_aci"] + BASELINES + ["oracle_hindsight"]
RL_LOAD_WEIGHT = 0.6
ETA_GRID = [0.01, 0.02, 0.05, 0.10, 0.20]  # full original grid (cheap at this data scale)
ETA_PRIMARY = 0.05
LOAD_WINDOW = 5      # requests, for the local-arrival-rate load proxy (original: 50)
RNG_GLOBAL = np.random.default_rng(20260825)

## Step 0: parse the trace rows into per-regime NumPy arrays\n\nThe original script reads `full_data_out_*.json` parts off disk with `glob`. Here we read the same per-row schema (`arrival_time`, `risk_score`, `slo_target`, `regime_label`, `function_id`, plus the binary violation `label`) directly out of the already-loaded `data` object. The per-regime sort-by-`arrival_time` logic is unchanged.

In [ ]:
def load_regime_arrays() -> dict[str, dict[str, np.ndarray]]:
    by_regime: dict[str, dict[str, list]] = {
        r: {"arrival_time": [], "risk_score": [], "slo_target": [], "label": [], "function_id": []} for r in REGIMES
    }
    examples = data["datasets"][0]["examples"]
    total = 0
    for e in examples:
        inp = json.loads(e["input"])
        r = inp["regime_label"]
        if r not in by_regime:
            raise ValueError(f"Unexpected regime_label {r!r} in dataset (row provenance mismatch)")
        by_regime[r]["arrival_time"].append(float(inp["arrival_time"]))
        by_regime[r]["risk_score"].append(float(inp["risk_score"]))
        by_regime[r]["slo_target"].append(float(inp["slo_target"]))
        by_regime[r]["label"].append(int(e["output"]))
        by_regime[r]["function_id"].append(inp["function_id"])
        total += 1
    logger.info(f"Loaded {total} rows from mini_demo_data.json")
    out = {}
    for r, cols in by_regime.items():
        idx_sorted = np.argsort(np.asarray(cols["arrival_time"]), kind="stable")
        out[r] = {
            "arrival_time": np.asarray(cols["arrival_time"])[idx_sorted],
            "risk_score": np.asarray(cols["risk_score"])[idx_sorted],
            "slo_target": np.asarray(cols["slo_target"])[idx_sorted],
            "label": np.asarray(cols["label"], dtype=bool)[idx_sorted],
            "function_id": np.asarray(cols["function_id"], dtype=object)[idx_sorted],
        }
    return out, total


regime_arrays, total_rows = load_regime_arrays()
for r in REGIMES:
    lab = regime_arrays[r]["label"]
    logger.info(f"regime={r}: n={len(lab)}, violation_rate={lab.mean():.4f}")

## Step 0b: value + load proxies, bootstrap-resampled seeded traces\n\nThe dataset has no native per-request `value` or `load` field, so the original script builds two documented proxies: `value` blends per-function SLO tightness with the per-request `risk_score`, and `load` is a local inverse-inter-arrival-time signal. Because the real trace has no native seed/replicate dimension, each "seed" is an i.i.d. bootstrap resample of that regime's rows, re-sorted by arrival time — this is what supplies the `N_SEEDS` needed for the over-seed bootstrap later.

In [ ]:
def value_proxy(slo_target: np.ndarray, risk_score: np.ndarray, global_median_inv_slo: float) -> np.ndarray:
    """Deterministic proxy: (1/slo_target) scaled by a per-request risk_score factor so the
    value signal varies at REQUEST granularity, not just per-function. Pure 1/slo_target alone
    is a per-function constant on this real trace (most regimes are dominated by a single
    function_id -- see VALIDITY_NOTES), which would make knapsack-vs-FCFS trivially
    indistinguishable from FCFS everywhere. risk_score is already a genuine per-request
    admission-time signal, so blending it in keeps the proxy request-specific and documented."""
    return (1.0 / slo_target) * (0.25 + 0.75 * risk_score) / global_median_inv_slo


def local_load_proxy(arrival_time: np.ndarray, window: int = LOAD_WINDOW) -> np.ndarray:
    n = len(arrival_time)
    load = np.zeros(n)
    for i in range(n):
        lo = max(0, i - window + 1)
        span = arrival_time[i] - arrival_time[lo]
        cnt = i - lo
        load[i] = (cnt / span) if span > 1e-9 else (load[i - 1] if i > 0 else 0.0)
    lo_v, hi_v = np.percentile(load, [1, 99])
    if hi_v <= lo_v:
        hi_v = lo_v + 1e-9
    return np.clip((load - lo_v) / (hi_v - lo_v), 0.0, 1.0)


def estimate_switch_index(function_id: np.ndarray) -> int:
    n = len(function_id)
    first = function_id[0]
    for i in range(1, n - 100):
        if function_id[i] != first:
            block = function_id[i : i + 100]
            if np.all(block != first):
                return i
    return n // 2


def make_seeded_trace(regime_arrays: dict[str, np.ndarray], seed: int, global_median_inv_slo: float) -> dict[str, np.ndarray]:
    n = len(regime_arrays["label"])
    rng = np.random.default_rng(seed * 1_000_003 + 17)
    resample_idx = rng.integers(0, n, size=n)
    order = np.argsort(regime_arrays["arrival_time"][resample_idx], kind="stable")
    idx = resample_idx[order]
    arrival_time = regime_arrays["arrival_time"][idx]
    risk_score = regime_arrays["risk_score"][idx]
    label = regime_arrays["label"][idx]
    slo_target = regime_arrays["slo_target"][idx]
    function_id = regime_arrays["function_id"][idx]
    load = local_load_proxy(arrival_time)
    return {
        "arrival_time": arrival_time,
        "score": risk_score,
        # composite_score is the nonconformity signal actually used by conformal_aci and
        # fixed_threshold: on this real trace, raw risk_score is NEAR-CONSTANT within most
        # regimes (single dominant function_id per regime -- see VALIDITY_NOTES), which would
        # collapse a score-only threshold into an all-or-nothing step function and defeat
        # ACI's online modulation entirely. Both risk_score and the load proxy are legitimate
        # admission-time-only signals, so blending them (0.5/0.5) restores genuine granularity
        # while staying faithful to the "admission-time features only" constraint.
        "composite_score": 0.5 * risk_score + 0.5 * load,
        "would_violate": label,
        "value": value_proxy(slo_target, risk_score, global_median_inv_slo),
        "load": load,
        "function_id": function_id,
    }

## The 5 admission policies\n\n`conformal_aci` is the proposed policy — it online-adjusts an admission threshold via an ACI-style update after every admitted request. The four comparisons are: a frozen `fixed_threshold` calibrated once offline, a frozen `index_based` load cap, a frozen `rl_frozen` load/score blend, and the non-causal `oracle_hindsight` that greedily admits the lowest-violation-probability requests per window subject to the alpha budget (an upper bound, not a deployable policy).

In [ ]:
def run_conformal_aci(stream: dict[str, np.ndarray], alpha: float, eta: float, tau0: float, key: str = "composite_score") -> np.ndarray:
    n = len(stream[key])
    dec = np.zeros(n, dtype=bool)
    score = stream[key]
    wviol = stream["would_violate"]
    tau = tau0
    for i in range(n):
        admit = score[i] <= tau
        dec[i] = admit
        if admit:
            tau += eta * (alpha - float(wviol[i]))
            if tau < 0.0:
                tau = 0.0
            elif tau > 1.0:
                tau = 1.0
    return dec


def calibrate_scalar_threshold(cal: dict[str, np.ndarray], key: str, target: float) -> float:
    lo, hi = 0.0, 1.0
    sig = cal[key]
    wviol = cal["would_violate"]
    for _ in range(30):
        mid = 0.5 * (lo + hi)
        admit = sig <= mid
        if admit.sum() == 0:
            lo = mid
            continue
        rate = wviol[admit].mean()
        if rate < target:
            lo = mid
        else:
            hi = mid
    return 0.5 * (lo + hi)


def run_index_based(stream: dict[str, np.ndarray], load_thresh: float) -> np.ndarray:
    return stream["load"] <= load_thresh


def run_rl_frozen(stream: dict[str, np.ndarray], mean_load: float, std_load: float, k: float) -> np.ndarray:
    combined = RL_LOAD_WEIGHT * stream["load"] + (1 - RL_LOAD_WEIGHT) * stream["score"]
    thresh = RL_LOAD_WEIGHT * mean_load + k * std_load
    return combined <= thresh


def run_oracle_hindsight(stream: dict[str, np.ndarray], alpha: float, window: int) -> np.ndarray:
    n = len(stream["would_violate"])
    dec = np.zeros(n, dtype=bool)
    lab = stream["would_violate"].astype(np.float64)
    score = stream["score"]
    for start in range(0, n, window):
        end = min(start + window, n)
        idx = np.arange(start, end)
        order = idx[np.lexsort((score[idx], lab[idx]))]
        cum = np.cumsum(lab[order])
        counts = np.arange(1, len(order) + 1)
        cum_mean = cum / counts
        k = int(np.sum(cum_mean <= alpha))
        dec[order[:k]] = True
    return dec

## Rolling stats over admitted requests, plus bootstrap/Holm helpers\n\n`admitted_rolling_rate` computes the rolling violation rate over the *admitted* subsequence only (a rejected request cannot violate its SLO). `mad_and_spike` reduces that trajectory to a single post-burn-in MAD-from-alpha and a max transient spike. `whole_seed_bootstrap_ci` and `holm_bonferroni` are the exact bootstrap/multiple-testing machinery used for the significance tests further down.

In [ ]:
def admitted_rolling_rate(dec: np.ndarray, wviol: np.ndarray, window: int) -> np.ndarray:
    admitted_labels = wviol[dec].astype(np.float64)
    n = len(admitted_labels)
    if n == 0:
        return np.array([])
    cs = np.cumsum(admitted_labels)
    win = np.empty(n)
    win[:window] = cs[:window] / np.arange(1, min(window, n) + 1)
    if n > window:
        win[window:] = (cs[window:] - cs[:-window]) / window
    return win


def mad_and_spike(rate: np.ndarray, burn_in: int) -> tuple[float, float]:
    post = rate[burn_in:]
    if len(post) == 0:
        return float("nan"), float("nan")
    dev = np.abs(post - ALPHA)
    return float(dev.mean()), float(dev.max())


def safe_float(x: Any) -> float | None:
    return None if (x is None or (isinstance(x, float) and np.isnan(x))) else float(x)


def ci95(samples: np.ndarray) -> list[float | None]:
    samples = np.asarray(samples, dtype=float)
    samples = samples[~np.isnan(samples)]
    if len(samples) == 0:
        return [None, None]
    return [float(np.percentile(samples, 2.5)), float(np.percentile(samples, 97.5))]


def holm_bonferroni(pvals: list[float]) -> list[float]:
    order = np.argsort(pvals)
    m = len(pvals)
    adj = np.empty(m)
    running_max = 0.0
    for rank, idx in enumerate(order):
        raw = pvals[idx] * (m - rank)
        running_max = max(running_max, raw)
        adj[idx] = min(running_max, 1.0)
    return adj.tolist()


def whole_seed_bootstrap_ci(per_seed_values: list[float], n_boot: int, rng: np.random.Generator) -> tuple[list[float | None], np.ndarray]:
    vals = np.asarray([v for v in per_seed_values if v is not None and not np.isnan(v)])
    if len(vals) == 0:
        return [None, None], np.array([])
    n = len(vals)
    picks = rng.integers(0, n, size=(n_boot, n))
    samples = vals[picks].mean(axis=1)
    return ci95(samples), samples


def burn_in_for(regime_n_total: int) -> int:
    return max(BURN_IN_MIN, int(round(BURN_IN_FRAC * regime_n_total)))

## Calibrate frozen baselines, then simulate all policies x regimes x seeds\n\nAll of `fixed_threshold`, `index_based`, and `rl_frozen` are calibrated ONCE on a held-out slice of the `stationary` regime — they never see the target alpha again after this point, unlike `conformal_aci`, which keeps adapting online. Then every policy is run over every regime and every bootstrap seed.

In [ ]:
def calibrate_all(regime_arrays: dict[str, dict[str, np.ndarray]], global_median_inv_slo: float) -> dict[str, Any]:
    n = len(regime_arrays["stationary"]["label"])
    rng = np.random.default_rng(CALIB_SEED)
    cal_idx = rng.choice(n, size=min(20_000, n), replace=False)
    cal_order = np.argsort(regime_arrays["stationary"]["arrival_time"][cal_idx], kind="stable")
    cal_idx = cal_idx[cal_order]
    cal = {
        "arrival_time": regime_arrays["stationary"]["arrival_time"][cal_idx],
        "score": regime_arrays["stationary"]["risk_score"][cal_idx],
        "would_violate": regime_arrays["stationary"]["label"][cal_idx],
        "value": value_proxy(regime_arrays["stationary"]["slo_target"][cal_idx], regime_arrays["stationary"]["risk_score"][cal_idx], global_median_inv_slo),
    }
    cal["load"] = local_load_proxy(cal["arrival_time"])
    cal["composite_score"] = 0.5 * cal["score"] + 0.5 * cal["load"]

    tau0_fixed = calibrate_scalar_threshold(cal, "composite_score", ALPHA)
    load_thresh_index = float(np.percentile(cal["load"], 70.0))  # frozen operational cap, NOT alpha-calibrated (misspecified by design)

    mean_load = float(cal["load"].mean())
    std_load = float(cal["load"].std()) or 1e-6
    best_k, best_diff = 0.0, np.inf
    for k in np.linspace(-6.0, 6.0, 481):
        dec = run_rl_frozen(cal, mean_load, std_load, k)
        if dec.sum() == 0:
            continue
        rate = cal["would_violate"][dec].mean()
        diff = abs(rate - ALPHA)
        if diff < best_diff:
            best_diff, best_k = diff, k

    logger.info(f"Calibrated on stationary(fold-mixed, n={len(cal_idx)}): tau0={tau0_fixed:.4f} load_thresh={load_thresh_index:.4f} rl_k={best_k:.3f}")
    return {
        "tau0_fixed": float(tau0_fixed),
        "load_thresh_index": float(load_thresh_index),
        "rl_k": float(best_k),
        "mean_load_stationary": mean_load,
        "std_load_stationary": std_load,
        "n_calibration_rows": int(len(cal_idx)),
    }


def simulate_policy_decisions(
    regime_arrays: dict[str, dict[str, np.ndarray]],
    calib: dict[str, Any],
    global_median_inv_slo: float,
    eta: float,
) -> dict[str, dict[str, dict[int, dict[str, np.ndarray]]]]:
    """logs[policy][regime][seed] -> {decision, would_violate, value, function_id}"""
    logs: dict[str, dict[str, dict[int, dict[str, np.ndarray]]]] = {p: {r: {} for r in REGIMES} for p in ALL_POLICIES}
    t0 = time.time()
    for regime in REGIMES:
        for seed in range(N_SEEDS):
            stream = make_seeded_trace(regime_arrays[regime], seed, global_median_inv_slo)
            dec_conf = run_conformal_aci(stream, ALPHA, eta, calib["tau0_fixed"])
            dec_fixed = stream["composite_score"] <= calib["tau0_fixed"]
            dec_index = run_index_based(stream, calib["load_thresh_index"])
            dec_rl = run_rl_frozen(stream, calib["mean_load_stationary"], calib["std_load_stationary"], calib["rl_k"])
            dec_oracle = run_oracle_hindsight(stream, ALPHA, WINDOW)
            for pname, dec in [
                ("conformal_aci", dec_conf),
                ("fixed_threshold", dec_fixed),
                ("index_based", dec_index),
                ("rl_frozen", dec_rl),
                ("oracle_hindsight", dec_oracle),
            ]:
                logs[pname][regime][seed] = {
                    "decision": dec,
                    "would_violate": stream["would_violate"],
                    "value": stream["value"],
                    "function_id": stream["function_id"],
                }
        logger.info(f"[eta={eta}] simulated regime={regime} for {N_SEEDS} seeds x {len(ALL_POLICIES)} policies ({time.time()-t0:.1f}s elapsed)")
    return logs


all_slo = np.concatenate([regime_arrays[r]["slo_target"] for r in REGIMES])
global_median_inv_slo = float(np.median(1.0 / all_slo))
boot_rng = np.random.default_rng(2026)
calib = calibrate_all(regime_arrays, global_median_inv_slo)
logs_primary = simulate_policy_decisions(regime_arrays, calib, global_median_inv_slo, ETA_PRIMARY)

## Per (policy, regime) deviation stats, plus Holm-corrected paired significance\n\nFor every `(policy, regime)` cell: average the rolling-rate trajectory across seeds, reduce to a post-burn-in MAD point estimate + bootstrap 95% CI, and flag pass/fail against the `TOL_PP` tolerance. Then, for each `(regime, baseline)` pair, bootstrap the paired MAD difference (`baseline - conformal`) across seeds and Holm-Bonferroni-correct the resulting p-values across all pairs.

In [ ]:
def compute_deviation_stats(
    logs: dict[str, Any], regime_arrays: dict[str, dict[str, np.ndarray]], boot_rng: np.random.Generator
) -> tuple[dict[str, Any], dict[str, dict[str, np.ndarray]]]:
    per_policy_regime: dict[str, dict[str, Any]] = {p: {} for p in ALL_POLICIES}
    rolling_series_for_plots: dict[str, dict[str, np.ndarray]] = {}

    for regime in REGIMES:
        rolling_series_for_plots[regime] = {}
        n_total = len(regime_arrays[regime]["label"])
        burn_in = burn_in_for(n_total)
        for pname in ALL_POLICIES:
            per_seed_mad, per_seed_spike, per_seed_admits = [], [], []
            rates_by_seed = []
            for seed in range(N_SEEDS):
                rec = logs[pname][regime][seed]
                rate = admitted_rolling_rate(rec["decision"], rec["would_violate"], WINDOW)
                rates_by_seed.append(rate)
                bi = min(burn_in, max(len(rate) - 1, 0))
                m, s = mad_and_spike(rate, bi)
                per_seed_mad.append(m)
                per_seed_spike.append(s)
                per_seed_admits.append(int(rec["decision"].sum()))
            maxlen = max((len(r) for r in rates_by_seed), default=0)
            if maxlen > 0:
                padded = np.full((N_SEEDS, maxlen), np.nan)
                for si, r in enumerate(rates_by_seed):
                    padded[si, : len(r)] = r
                with np.errstate(invalid="ignore"):
                    rolling_series_for_plots[regime][pname] = np.nanmean(padded, axis=0)
            else:
                rolling_series_for_plots[regime][pname] = np.array([])

            mad_ci, mad_samples = whole_seed_bootstrap_ci(per_seed_mad, N_BOOTSTRAP, boot_rng)
            spike_ci, spike_samples = whole_seed_bootstrap_ci(per_seed_spike, N_BOOTSTRAP, boot_rng)
            mad_valid = [m for m in per_seed_mad if not np.isnan(m)]
            mad_point = float(np.mean(mad_valid)) if mad_valid else float("nan")
            spike_valid = [s for s in per_seed_spike if not np.isnan(s)]
            spike_point = float(np.max(spike_valid)) if spike_valid else float("nan")
            insufficient = bool(sum(per_seed_admits) < WINDOW // 2)

            entry = {
                "mad_point": safe_float(mad_point),
                "mad_ci95": mad_ci,
                "max_spike_point": safe_float(spike_point),
                "max_spike_ci95": spike_ci,
                "n_seeds": N_SEEDS,
                "total_admits_across_seeds": int(sum(per_seed_admits)),
                "per_seed_admits": per_seed_admits,
                "burn_in_admitted_requests": burn_in,
                "insufficient_admissions": insufficient,
                "bootstrap_method": "over_seed_resample_with_replacement",
                "n_bootstrap": N_BOOTSTRAP,
                "tolerance_pass_3pp": bool((not insufficient) and (not np.isnan(mad_point)) and mad_point <= TOL_PP),
            }
            if regime == "regime_switch":
                switch_idx_full = estimate_switch_index(regime_arrays[regime]["function_id"])
                entry["estimated_switch_index_in_full_trace"] = int(switch_idx_full)
            per_policy_regime[pname][regime] = entry
            gc.collect()
        logger.info(f"[regime={regime}] deviation stats done for {len(ALL_POLICIES)} policies (burn_in={burn_in})")
    return per_policy_regime, rolling_series_for_plots


def compute_paired_significance(logs: dict[str, Any], regime_arrays: dict[str, Any], boot_rng: np.random.Generator) -> list[dict[str, Any]]:
    pair_records = []
    for regime in REGIMES:
        n_total = len(regime_arrays[regime]["label"])
        burn_in = burn_in_for(n_total)
        for baseline in BASELINES:
            mad_c_seed, mad_b_seed = [], []
            for seed in range(N_SEEDS):
                rc = logs["conformal_aci"][regime][seed]
                rb = logs[baseline][regime][seed]
                rate_c = admitted_rolling_rate(rc["decision"], rc["would_violate"], WINDOW)
                rate_b = admitted_rolling_rate(rb["decision"], rb["would_violate"], WINDOW)
                bi_c = min(burn_in, max(len(rate_c) - 1, 0))
                bi_b = min(burn_in, max(len(rate_b) - 1, 0))
                m_c, _ = mad_and_spike(rate_c, bi_c)
                m_b, _ = mad_and_spike(rate_b, bi_b)
                mad_c_seed.append(m_c)
                mad_b_seed.append(m_b)
            paired_diff = np.array(
                [b - c for b, c in zip(mad_b_seed, mad_c_seed) if not (np.isnan(b) or np.isnan(c))]
            )  # >0 => baseline deviates more => conformal better
            insufficient_pair = len(paired_diff) < 3
            if insufficient_pair:
                pair_records.append(
                    {"regime": regime, "baseline": baseline, "paired_diff_ci95": [None, None], "p_boot": None, "insufficient_admissions": True}
                )
                continue
            n = len(paired_diff)
            picks = boot_rng.integers(0, n, size=(N_BOOTSTRAP, n))
            boot_means = paired_diff[picks].mean(axis=1)
            lo, hi = ci95(boot_means)
            p_boot = float(2 * min((boot_means <= 0).mean(), (boot_means >= 0).mean()))
            pair_records.append(
                {
                    "regime": regime,
                    "baseline": baseline,
                    "paired_diff_ci95": [lo, hi],
                    "p_boot": p_boot,
                    "insufficient_admissions": False,
                    "n_seed_pairs": int(n),
                }
            )
            del picks, boot_means
            gc.collect()
    holm_pvals = [r["p_boot"] if r["p_boot"] is not None else 1.0 for r in pair_records]
    holm_adj = holm_bonferroni(holm_pvals)
    for r, p_adj in zip(pair_records, holm_adj):
        r["p_holm"] = None if r["p_boot"] is None else p_adj
        r["conformal_significantly_better"] = bool(
            (not r.get("insufficient_admissions", False))
            and r["paired_diff_ci95"][0] is not None
            and r["paired_diff_ci95"][0] > 0
            and r["p_holm"] is not None
            and r["p_holm"] < 0.05
        )
    logger.info(f"Paired significance tests: {len(pair_records)} (regime x baseline), Holm-corrected, over-seed resample")
    return pair_records


per_policy_regime, rolling_series = compute_deviation_stats(logs_primary, regime_arrays, boot_rng)
pair_records = compute_paired_significance(logs_primary, regime_arrays, boot_rng)

## Matched-violation-rate value comparison, and knapsack-vs-FCFS\n\nTo compare *value*, not just violation rate, each baseline is re-thresholded to match conformal-ACI's realized violation rate on `stationary`, then the total admitted `value` is compared. Separately, a value-aware knapsack admission rule is compared against plain FCFS among conformal-eligible requests on `regime_switch` (chosen over `stationary` because `stationary` here is dominated by a single `function_id`, making the value proxy degenerate there).

In [ ]:
def rethreshold_scalar(stream: dict[str, np.ndarray], key: str, target: float) -> tuple[float, np.ndarray]:
    tau = calibrate_scalar_threshold(stream, key, target)
    return tau, (stream[key] <= tau)


def rethreshold_rl(stream: dict[str, np.ndarray], mean_load: float, std_load: float, target: float) -> tuple[float, np.ndarray]:
    best_k, best_diff, best_dec = 0.0, np.inf, np.zeros(len(stream["composite_score"]), dtype=bool)
    for k in np.linspace(-6.0, 6.0, 241):
        dec = run_rl_frozen(stream, mean_load, std_load, k)
        if dec.sum() == 0:
            continue
        diff = abs(stream["would_violate"][dec].mean() - target)
        if diff < best_diff:
            best_diff, best_k, best_dec = diff, k, dec
    return best_k, best_dec


def matched_value_and_knapsack(
    regime_arrays: dict[str, Any], calib: dict[str, Any], logs_primary: dict[str, Any], global_median_inv_slo: float, boot_rng: np.random.Generator
) -> tuple[dict[str, Any], dict[str, Any]]:
    seed0 = 0
    stream = make_seeded_trace(regime_arrays["stationary"], seed0, global_median_inv_slo)
    dec_conf = logs_primary["conformal_aci"]["stationary"][seed0]["decision"]
    conf_rate = float(stream["would_violate"][dec_conf].sum() / max(dec_conf.sum(), 1))
    total_value_conformal = float(stream["value"][dec_conf].sum())

    value_gap: dict[str, Any] = {}
    for baseline in BASELINES + ["oracle_hindsight"]:
        if baseline == "fixed_threshold":
            tau, dec_matched = rethreshold_scalar(stream, "composite_score", conf_rate)
            method = f"bisection re-threshold on composite_score; tau={tau:.4f}"
        elif baseline == "index_based":
            tau, dec_matched = rethreshold_scalar(stream, "load", conf_rate)
            method = f"bisection re-threshold on load proxy; tau={tau:.4f}"
        elif baseline == "rl_frozen":
            k, dec_matched = rethreshold_rl(stream, calib["mean_load_stationary"], calib["std_load_stationary"], conf_rate)
            method = f"bisection re-search over frozen boundary width k; k={k:.4f}"
        else:
            dec_matched = logs_primary["oracle_hindsight"]["stationary"][seed0]["decision"]
            method = "hindsight-optimal oracle already targets alpha per window by construction"

        total_value_matched = float(stream["value"][dec_matched].sum())
        realized_rate_matched = float(stream["would_violate"][dec_matched].sum() / max(dec_matched.sum(), 1))
        gap_pct = (total_value_matched - total_value_conformal) / total_value_matched * 100 if total_value_matched > 0 else float("nan")

        n = len(stream["value"])
        picks = boot_rng.integers(0, n, size=(N_BOOTSTRAP, min(n, 20_000)))
        val_conf = stream["value"] * dec_conf.astype(float)
        val_match = stream["value"] * dec_matched.astype(float)
        tv_conf = val_conf[picks].sum(axis=1) * (n / picks.shape[1])
        tv_match = val_match[picks].sum(axis=1) * (n / picks.shape[1])
        with np.errstate(invalid="ignore", divide="ignore"):
            gap_samples = np.where(tv_match > 0, (tv_match - tv_conf) / tv_match * 100, np.nan)
        gap_ci = ci95(gap_samples)
        degenerate = bool(total_value_matched < 0.05 * total_value_conformal)

        value_gap[baseline] = {
            "rethreshold_method": method,
            "target_violation_rate_matched_pct": round(conf_rate * 100, 3),
            "realized_violation_rate_matched_pct": round(realized_rate_matched * 100, 3),
            "total_value_conformal": total_value_conformal,
            "total_value_baseline_matched": total_value_matched,
            "value_gap_pct": safe_float(gap_pct),
            "value_gap_pct_ci95": gap_ci,
            "degenerate_matched_denominator": degenerate,
            "disconfirmed_over_50pct_loss": bool(
                (not degenerate) and (not np.isnan(gap_pct)) and gap_pct > 50 and gap_ci[0] is not None and gap_ci[0] > 50
            ),
        }
        del picks, tv_conf, tv_match
        gc.collect()
    logger.info("Matched-violation-rate value comparison (stationary) computed for all baselines")

    KNAPSACK_REGIME = "regime_switch"
    stream_knap = make_seeded_trace(regime_arrays[KNAPSACK_REGIME], 0, global_median_inv_slo)
    eligible = stream_knap["composite_score"] <= calib["tau0_fixed"]
    n = len(eligible)
    dec_fcfs = np.zeros(n, dtype=bool)
    dec_knap = np.zeros(n, dtype=bool)
    capacity_frac = 0.55
    for start in range(0, n, WINDOW):
        end = min(start + WINDOW, n)
        idx = np.arange(start, end)
        elig_idx = idx[eligible[idx]]
        cap = min(int(round(capacity_frac * len(idx))), len(elig_idx))
        dec_fcfs[elig_idx[:cap]] = True
        if cap > 0:
            order = elig_idx[np.argsort(-stream_knap["value"][elig_idx])]
            dec_knap[order[:cap]] = True

    rate_fcfs = admitted_rolling_rate(dec_fcfs, stream_knap["would_violate"], WINDOW)
    rate_knap = admitted_rolling_rate(dec_knap, stream_knap["would_violate"], WINDOW)
    bi = burn_in_for(n)
    mad_fcfs, _ = mad_and_spike(rate_fcfs, min(bi, max(len(rate_fcfs) - 1, 0)))
    mad_knap, _ = mad_and_spike(rate_knap, min(bi, max(len(rate_knap) - 1, 0)))

    picks = boot_rng.integers(0, n, size=(N_BOOTSTRAP, min(n, 20_000)))
    scale = n / picks.shape[1]
    wv = stream_knap["would_violate"].astype(float)
    dec_fcfs_f, dec_knap_f = dec_fcfs.astype(float), dec_knap.astype(float)
    rate_fcfs_boot = (dec_fcfs_f[picks] * wv[picks]).sum(axis=1) / np.maximum(dec_fcfs_f[picks].sum(axis=1), 1)
    rate_knap_boot = (dec_knap_f[picks] * wv[picks]).sum(axis=1) / np.maximum(dec_knap_f[picks].sum(axis=1), 1)
    mad_diff_samples = np.abs(rate_knap_boot - ALPHA) - np.abs(rate_fcfs_boot - ALPHA)
    mad_diff_ci = ci95(mad_diff_samples)

    val_fcfs = (stream_knap["value"] * dec_fcfs_f)
    val_knap = (stream_knap["value"] * dec_knap_f)
    vg_fcfs = val_fcfs[picks].sum(axis=1) * scale
    vg_knap = val_knap[picks].sum(axis=1) * scale
    value_gain_ci = ci95(vg_knap - vg_fcfs)

    knapsack_check = {
        "regime_used": KNAPSACK_REGIME,
        "capacity_frac": capacity_frac,
        "mad_fcfs": safe_float(mad_fcfs),
        "mad_knapsack": safe_float(mad_knap),
        "mad_diff_ci95_knapsack_minus_fcfs": mad_diff_ci,
        "guarantee_indistinguishable": bool(mad_diff_ci[0] is not None and mad_diff_ci[0] <= 0 <= mad_diff_ci[1]),
        "total_value_fcfs": float(val_fcfs.sum()),
        "total_value_knapsack": float(val_knap.sum()),
        "value_gain_ci95": value_gain_ci,
        "value_gain_significant_and_positive": bool(value_gain_ci[0] is not None and value_gain_ci[0] > 0),
    }
    logger.info(f"Knapsack vs FCFS: mad_diff_ci={mad_diff_ci}, value_gain_ci={value_gain_ci}")
    return value_gap, knapsack_check


value_gap, knapsack_check = matched_value_and_knapsack(regime_arrays, calib, logs_primary, global_median_inv_slo, boot_rng)

## Eta sensitivity\n\nSweeps `conformal_aci`'s adaptation rate `eta` over `ETA_GRID` to show the tradeoff between tracking speed (lower MAD) and transient overshoot (max spike), for `stationary`, `regime_switch`, and `adversarial`.

In [ ]:
def eta_sensitivity(
    regime_arrays: dict[str, Any], calib: dict[str, Any], global_median_inv_slo: float
) -> dict[str, Any]:
    result: dict[str, Any] = {}
    target_regimes = ["stationary", "regime_switch", "adversarial"]
    for regime in target_regimes:
        result[regime] = {}
        n_total = len(regime_arrays[regime]["label"])
        burn_in = burn_in_for(n_total)
        for eta in ETA_GRID:
            mads, spikes = [], []
            for seed in range(N_SEEDS):
                stream = make_seeded_trace(regime_arrays[regime], seed, global_median_inv_slo)
                dec = run_conformal_aci(stream, ALPHA, eta, calib["tau0_fixed"])
                rate = admitted_rolling_rate(dec, stream["would_violate"], WINDOW)
                bi = min(burn_in, max(len(rate) - 1, 0))
                m, s = mad_and_spike(rate, bi)
                mads.append(m)
                spikes.append(s)
            mads_v = [m for m in mads if not np.isnan(m)]
            spikes_v = [s for s in spikes if not np.isnan(s)]
            result[regime][str(eta)] = {
                "mad_mean_over_seeds": safe_float(np.mean(mads_v)) if mads_v else None,
                "max_spike_mean_over_seeds": safe_float(np.mean(spikes_v)) if spikes_v else None,
            }
        logger.info(f"Eta sensitivity done for regime={regime}")
    return result


eta_sens = eta_sensitivity(regime_arrays, calib, global_median_inv_slo)

## Overall verdict\n\nCombines the tolerance pass/fail flags, the Holm-corrected significant-pairs fraction, and the matched-value disconfirmation check into a single `CONFIRMED` / `PARTIALLY_CONFIRMED` / `DISCONFIRMED` verdict, exactly as the original script does.

In [ ]:
tolerance_all_pass = all(per_policy_regime["conformal_aci"][r]["tolerance_pass_3pp"] for r in REGIMES)
sig_pairs_pass = [r for r in pair_records if r["conformal_significantly_better"]]
sig_frac = len(sig_pairs_pass) / len(pair_records) if pair_records else 0.0
any_value_disconfirm = any(v["disconfirmed_over_50pct_loss"] for k, v in value_gap.items() if k in BASELINES)

if tolerance_all_pass and sig_frac >= 0.75 and not any_value_disconfirm:
    overall_verdict = "CONFIRMED"
    justification = (
        f"On the REAL Azure-trace dataset, conformal-ACI's MAD stayed within the pre-registered "
        f"{TOL_PP*100:.0f}pp tolerance of alpha in all {len(REGIMES)} regimes; it was Holm-corrected "
        f"significantly better than baselines in {len(sig_pairs_pass)}/{len(pair_records)} (regime,baseline) "
        f"pairs (>=75% threshold); no baseline's matched-value gap exceeded the 50% disconfirming threshold."
    )
elif not tolerance_all_pass and sig_frac < 0.25:
    overall_verdict = "DISCONFIRMED"
    justification = (
        f"On the real-trace data, conformal-ACI failed the {TOL_PP*100:.0f}pp tolerance criterion in at "
        f"least one regime AND was Holm-corrected significantly better than baselines in fewer than 25% "
        f"of pairs ({len(sig_pairs_pass)}/{len(pair_records)})."
    )
elif any_value_disconfirm:
    overall_verdict = "DISCONFIRMED"
    disconf_names = [k for k, v in value_gap.items() if k in BASELINES and v["disconfirmed_over_50pct_loss"]]
    justification = f"Matched-violation-rate value comparison shows conformal-ACI losing >50% value vs {disconf_names}, CI lower bound also >50%."
else:
    overall_verdict = "PARTIALLY_CONFIRMED"
    justification = (
        f"Tolerance pass across all regimes: {tolerance_all_pass}. Significant-better fraction: {sig_frac:.2f} "
        f"of {len(pair_records)} pairs. No baseline value comparison crossed the 50% disconfirming threshold."
    )

logger.info(f"OVERALL VERDICT: {overall_verdict}")
logger.info(justification)

## Results\n\nA readable summary table of the per-(policy, regime) MAD/spike numbers, plus the rolling violation-rate trajectories and the eta-sensitivity curves — the same two figure types the original script writes to `figures/`.

In [ ]:
print(f"OVERALL VERDICT: {overall_verdict}\n")
print(justification, "\n")

header = f"{'policy':<16}{'regime':<15}{'MAD':>8}{'max_spike':>11}{'tol_pass_3pp':>14}"
print(header)
print("-" * len(header))
for pname in ALL_POLICIES:
    for regime in REGIMES:
        e = per_policy_regime[pname][regime]
        mad = "nan" if e["mad_point"] is None else f"{e['mad_point']:.4f}"
        spike = "nan" if e["max_spike_point"] is None else f"{e['max_spike_point']:.4f}"
        print(f"{pname:<16}{regime:<15}{mad:>8}{spike:>11}{str(e['tolerance_pass_3pp']):>14}")

print("\nHolm-corrected paired significance (conformal_aci vs baseline):")
for r in pair_records:
    p_holm = "n/a" if r["p_holm"] is None else f"{r['p_holm']:.3f}"
    print(f"  {r['regime']:<15}{r['baseline']:<16} p_holm={p_holm:<8} sig_better={r.get('conformal_significantly_better')}")

# --- Rolling violation-rate trajectories, one panel per regime ---
colors = {
    "conformal_aci": "tab:blue",
    "fixed_threshold": "tab:orange",
    "index_based": "tab:green",
    "rl_frozen": "tab:red",
    "oracle_hindsight": "tab:gray",
}
band = (ALPHA - TOL_PP, ALPHA + TOL_PP)
fig, axes = plt.subplots(len(REGIMES), 1, figsize=(9, 3.2 * len(REGIMES)))
for ax, regime in zip(axes, REGIMES):
    ax.axhspan(band[0], band[1], color="lightgray", alpha=0.5, label=f"+/-{TOL_PP*100:.0f}pp tolerance band")
    ax.axhline(ALPHA, color="black", linestyle="--", linewidth=1, label=f"alpha={ALPHA}")
    for pname in ALL_POLICIES:
        series = rolling_series[regime][pname]
        if len(series):
            ax.plot(series, label=pname, color=colors[pname], linewidth=1.3, alpha=0.9, marker="o", markersize=3)
    ax.set_xlabel("admitted-request index")
    ax.set_ylabel("rolling violation rate")
    ax.set_title(f"regime={regime}")
    ax.set_ylim(-0.05, 1.05)
ax.legend(loc="upper right", fontsize=7, ncol=2)
fig.suptitle(f"Rolling SLO-violation rate vs alpha (window={WINDOW} admitted requests, demo scale)")
fig.tight_layout()
plt.show()

# --- Eta sensitivity ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for regime in eta_sens:
    etas = sorted(eta_sens[regime].keys(), key=float)
    mads = [eta_sens[regime][e]["mad_mean_over_seeds"] for e in etas]
    spikes = [eta_sens[regime][e]["max_spike_mean_over_seeds"] for e in etas]
    axes[0].plot([float(e) for e in etas], mads, marker="o", label=regime)
    axes[1].plot([float(e) for e in etas], spikes, marker="o", label=regime)
axes[0].axhline(TOL_PP, color="red", linestyle=":", label=f"{TOL_PP*100:.0f}pp tolerance")
axes[0].set_xlabel("eta"); axes[0].set_ylabel("MAD (post burn-in)"); axes[0].set_title("MAD vs eta"); axes[0].legend(fontsize=7)
axes[1].set_xlabel("eta"); axes[1].set_ylabel("max transient spike"); axes[1].set_title("Max spike vs eta"); axes[1].legend(fontsize=7)
fig.tight_layout()
plt.show()